In [2]:


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import random
import matplotlib.pyplot as plt
import math     

torch.__version__
torchvision.__version__
import brevitas
torch.cuda.is_available()    
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device}")   
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)       

print(F)


Using Device: cuda
<module 'torch.nn.functional' from '/home/fede/miniconda3/envs/ptorch/lib/python3.9/site-packages/torch/nn/functional.py'>


In [3]:
from brevitas.core.quant import QuantType
from brevitas.core.restrict_val import RestrictValueType
from brevitas.core.scaling import ScalingImplType

W = int(8)   # total bits
I = int(1)   # integer bits including sign
Fr = W - I

In [4]:
# This is a simple implementation of a quantized Vision Transformer (ViT) using Brevitas.
# It demonstrates how to use Brevitas for quantizing the weights and activations of a ViT model.
# First, we need to install the required libraries.
from brevitas.nn import QuantConv2d, QuantLinear, QuantReLU 
from brevitas.quant import Int8ActPerTensorFloat, Int8WeightPerTensorFloat
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [ ]:
# The Vision Transformer is composed of several components:
# 1. Patch Embedding: Divides the input image into patches and embeds them into a lower-dimensional space.
# 2. Multi-Head Self-Attention: Computes attention scores for each patch and aggregates information from all patches.
# 3. Feed-Forward Neural Network: Applies a feed-forward network to each patch independently.
# 4. Layer Normalization and Residual Connections: Normalizes the output and adds skip connections.

In [5]:
config = {
    "batch_size":32,
    "patch_size": 8,
    "mlp_size": 192*4,
    "embed_dim": 192,
    "num_heads": 4,
    "dropout": 0.1,
    "image_size": 32,
    "num_classes": 10,
    "num_channels": 1,
    "qkv_bias": True,
    "depth": 1,
    "epochs": 10
}

In [6]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

config_cifar = {
    "data_root": "./data",
    "batch_size": 128,
    "num_workers": 4,
}

# Common CIFAR normalization often used in practice
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

train_dataset = datasets.CIFAR100(
    root=config_cifar["data_root"],
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = datasets.CIFAR100(
    root=config_cifar["data_root"],
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config_cifar["batch_size"],
    shuffle=True,
    num_workers=config_cifar["num_workers"],
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config_cifar["batch_size"],
    shuffle=False,
    num_workers=config_cifar["num_workers"],
    pin_memory=True
)

Files already downloaded and verified
Files already downloaded and verified


In [7]:
# For test/val
transform_test = transforms.Compose([
    transforms.Pad(2),          # 28x28 -> 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

# For training (with augmentations)
transform_train = transforms.Compose([
    transforms.Pad(2),          # 28x28 -> 32x32
    #transforms.RandomCrop(28, padding=4),
    transforms.RandomHorizontalFlip(),
    #transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"])
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"])

In [8]:
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

def test(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

In [9]:
from brevitas.quant import Int8BiasPerTensorFixedPointInternalScaling, Int8WeightPerTensorFixedPoint, Int8WeightPerTensorFloat
from brevitas.quant import Int8ActPerTensorFixedPoint, Int8Bias, Uint8ActPerTensorFloat, Int32Bias
from brevitas.nn import QuantIdentity


class QuantPatchEmbeddings(nn.Module):
    """
    Convert the image into patches and then project them into a vector space.
    """

    def __init__(self, config):
        super().__init__()
        self.image_size = config["image_size"]
        self.patch_size = config["patch_size"]
        self.num_channels = config["num_channels"]
        self.embed_dim = config["embed_dim"]
        self.drop = nn.Dropout(config["dropout"])
        # Calculate the number of patches from the image size and patch size
        self.num_patches = (self.image_size // self.patch_size) ** 2
        
        # Create a projection layer to convert the image into patches
        # The layer projects each patch into a vector of size hidden_size
        
        self.proj = QuantConv2d(
            self.num_channels, 
            self.embed_dim, 
            kernel_size=self.patch_size, 
            stride=self.patch_size,
            bias=True,
            input_quant= Int8ActPerTensorFloat,
            weight_quant = Int8WeightPerTensorFloat,
            bias_quant=Int32Bias,
            output_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True
                            )
        
        # --- CLS Token: Standard Parameter ---
        self.cls_token_param = nn.Parameter(torch.zeros(1, 1, self.embed_dim))

                # --- Positional Embedding: Standard Parameter ---
        self.pos_embed_param = nn.Parameter(torch.zeros(1, 1 + self.num_patches, self.embed_dim))

        # --- CLS Token: Quantizer Wrapper ---
        self.cls_token_quant = QuantIdentity(
            act_quant=self.proj.output_quant,return_quant_tensor=True)
        
        # --- Positional Embedding: Quantizer Wrapper ---
        self.pos_embed_quant = QuantIdentity(
            act_quant=self.proj.output_quant,return_quant_tensor=True)

        nn.init.trunc_normal_(self.cls_token_param, std=0.02)
        nn.init.trunc_normal_(self.pos_embed_param, std=0.02)

        self.out_quant = QuantIdentity(
            act_quant=self.proj.output_quant,
            return_quant_tensor=True
        )


    def forward(self, x):
        B = x.shape[0]

        # Project image patches: [B, C, H, W] -> [B, embed_dim, H/P, W/P]
        x = self.proj(x)                  # [B, E, H', W']
        x = x.flatten(2).transpose(1, 2)  # [B, N, E]

        # Expand CLS token to batch size and prepend to patch tokens
        cls_token = self.cls_token_param.expand(B, -1, -1)
        cls_token = self.cls_token_quant(cls_token)  # [B, 1, E]
        x = torch.cat((cls_token, x), dim=1)          # [B, N+1, E]

        positional = self.pos_embed_quant(self.pos_embed_param)

        # Add positional embedding
        x = x + positional

        

        x = self.drop(x)


        return self.out_quant(x)

In [10]:
class QuantMLP(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.fc1 = QuantLinear(
            config["embed_dim"], 
            config["mlp_size"],
            bias=True,
            input_quant= Int8ActPerTensorFloat,
            weight_quant = Int8WeightPerTensorFloat,
            bias_quant=Int32Bias,
            output_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True
        )
        self.act = QuantReLU(
            act_quant= Uint8ActPerTensorFloat,
            return_quant_tensor=True
        )
        self.drop = nn.Dropout(config["dropout"])
        self.fc2 = QuantLinear(
            config["mlp_size"], 
            config["embed_dim"],
            bias=True,
            #same quantizer for both FC2 Input -- FC1 Output
            input_quant= self.fc1.output_quant,
            weight_quant = Int8WeightPerTensorFloat,
            bias_quant=Int32Bias,
            output_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True
        )

        #From the paper https://openaccess.thecvf.com/content/ICCV2021W/NeurArch/papers/Yao_Leveraging_Batch_Normalization_for_Vision_Transformers_ICCVW_2021_paper.pdf
        self.norm = nn.BatchNorm1d(config["mlp_size"])

    def _bn(self, bn, x):              # x: [B, N, D]
        x = x.transpose(1, 2)          # [B, D, N]
        x = bn(x)                      # BN over features
        return x.transpose(1, 2)       # back to [B, N, D]

    def forward(self,x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)

        #x = self._bn(self.norm, x)

        x = self.fc2(x)
        x = self.drop(x)

        return x

In [11]:
import torch
import torch.nn.functional as F


def streaming_partial_softmax_torch(
    x: torch.Tensor,
    integerize: bool = True,
    width: int = 16,
    return_uint8: bool = False,
) -> torch.Tensor:

    if x.dim() != 4:
        raise ValueError(f"Expected x with shape [B, H, N, N], got {tuple(x.shape)}")

    Bsz, n_heads, seq_length, seq_length_2 = x.shape

    if seq_length != seq_length_2:
        raise ValueError(f"Expected square attention matrix, got {seq_length} x {seq_length_2}")

    if not integerize:
        return torch.softmax(x, dim=-1)

    original_seq_length = seq_length

    # Pad to next multiple of width
    padded_seq_length = ((seq_length + width - 1) // width) * width
    pad_amount = padded_seq_length - seq_length

    if pad_amount > 0:
        # Pad columns with very negative values so they contribute ~0 to softmax
        x = F.pad(x, pad=(0, pad_amount), mode="constant", value=-128.0)

        # Also pad rows, because this implementation expects square [N, N]
        x = F.pad(x, pad=(0, 0, 0, pad_amount), mode="constant", value=-128.0)

    Bsz, n_heads, seq_length, _ = x.shape
    groups = seq_length // width

    B_bits = 8
    range_scale = 32
    eps_max = range_scale * B_bits / (2 ** B_bits)

    x_int = torch.round(x).clamp(-128, 127).to(torch.int64)

    exp_partial_sum = torch.zeros(
        Bsz, n_heads, seq_length,
        device=x.device,
        dtype=torch.int64,
    )

    global_max = torch.full(
        (Bsz, n_heads, seq_length),
        -128,
        device=x.device,
        dtype=torch.int64,
    )

    for i in range(groups):
        start = i * width
        end = start + width

        block = x_int[..., start:end]

        current_max = block.max(dim=-1).values

        max_shift = torch.floor(
            (current_max - global_max).float() * eps_max
            + 0.5
            + torch.finfo(torch.float32).eps
        ).to(torch.int64)

        mask = current_max > global_max

        shift_sum = torch.where(
            mask,
            max_shift,
            torch.zeros_like(max_shift),
        )

        global_max = torch.where(mask, current_max, global_max)

        diff = global_max.unsqueeze(-1) - block

        shift = torch.floor(
            diff.float() * eps_max
            + 0.5
            + torch.finfo(torch.float32).eps
        ).to(torch.int64)

        shift = shift.clamp(min=0)

        base = torch.full_like(shift, 2 ** 8)
        exp_block = torch.bitwise_right_shift(base, shift)

        exp_sum = exp_block.sum(dim=-1)

        exp_partial_sum = (
            torch.bitwise_right_shift(exp_partial_sum, shift_sum.clamp(min=0))
            + exp_sum
        )

    exp_partial_sum = exp_partial_sum.clamp(min=1)

    exp_partial_sum_inverse = ((2 ** 8 - 1) * (2 ** 8)) // exp_partial_sum

    diff = global_max.unsqueeze(-1) - x_int

    shift = torch.floor(
        diff.float() * eps_max
        + 0.5
        + torch.finfo(torch.float32).eps
    ).to(torch.int64)

    shift = shift.clamp(min=0)

    attn_uint8 = torch.bitwise_right_shift(
        exp_partial_sum_inverse.unsqueeze(-1),
        shift,
    ).clamp(0, 255).to(torch.uint8)

    # Crop back to original [B, H, N, N]
    attn_uint8 = attn_uint8[..., :original_seq_length, :original_seq_length]

    if return_uint8:
        return attn_uint8

    return attn_uint8.to(dtype=x.dtype) / 255.0



In [12]:
import math
import torch
import torch.nn as nn

from brevitas.nn import QuantLinear, QuantIdentity

try:
    from brevitas.quant import (
        Int8ActPerTensorFloat,
        Int8WeightPerTensorFloat,
        Int32Bias,
    )
except ImportError:
    from brevitas.quant.scaled_int import (
        Int8ActPerTensorFloat,
        Int8WeightPerTensorFloat,
        Int32Bias,
    )


# ---------------------------------------------------------------------
# HLS EXP LUT
# ---------------------------------------------------------------------

EXP_LUT_64 = torch.tensor([
    1,   1,   1,   1,   1,   1,   1,   1,
    1,   1,   1,   1,   1,   1,   2,   2,
    2,   2,   2,   2,   2,   2,   2,   3,
    3,   3,   3,   4,   4,   4,   5,   5,
    6,   7,   7,   8,   9,   10,  12,  13,
    15,  17,  19,  21,  24,  27,  30,  34,
    39,  44,  50,  57,  64,  73,  82,  93,
    106, 120, 136, 154, 175, 199, 226, 255
], dtype=torch.int32)


def _value(x):
    """
    Return the floating-point tensor carried by a Brevitas QuantTensor,
    or x itself if x is already a torch.Tensor.
    """
    return x.value if hasattr(x, "value") else x


def _quant_tensor_to_int(x, bit_width=8, signed=True):
    """
    Convert a Brevitas QuantTensor to its integer-domain representation.

    If x is not a QuantTensor, this simply rounds it.
    """

    if hasattr(x, "value") and hasattr(x, "scale"):
        value = x.value
        scale = x.scale

        if hasattr(x, "zero_point") and x.zero_point is not None:
            zero_point = x.zero_point
        else:
            zero_point = 0.0

        x_int = torch.round(value / scale + zero_point)
    else:
        x_int = torch.round(x)

    if signed:
        qmin = -(2 ** (bit_width - 1))
        qmax = 2 ** (bit_width - 1) - 1
    else:
        qmin = 0
        qmax = 2 ** bit_width - 1

    x_int = torch.clamp(x_int, qmin, qmax)

    return x_int.to(torch.int32)


def hls_exp_lut(z):
    """
    PyTorch equivalent of your HLS exp_lut_lookup_int.

    HLS behavior:
        z_clip = clamp(z, -255, 0)
        idx = (z_clip + 255) >> 2
        return exp_lut[idx]
    """

    lut = EXP_LUT_64.to(device=z.device)

    z = z.to(torch.int32)
    z = torch.clamp(z, min=-255, max=0)

    idx = (z + 255) // 4

    return lut[idx]


def hls_lut_softmax(logits_int, softmax_scale=127):
    """
    HLS-like LUT softmax.

    Args:
        logits_int:
            Integer tensor. Softmax is computed over the last dimension.

        softmax_scale:
            Use 127 if your HLS output is interpreted as signed int8.
            Use 255 if your HLS output is interpreted as unsigned uint8.

    Returns:
        Floating tensor approximately in [0, 1].
    """

    logits_int = logits_int.to(torch.int32)

    row_max = logits_int.max(dim=-1, keepdim=True).values

    z = logits_int - row_max

    exp_int = hls_exp_lut(z)

    row_sum = exp_int.sum(dim=-1, keepdim=True).clamp(min=1)

    y_int = (exp_int * softmax_scale) // row_sum

    y_float = y_int.to(torch.float32) / float(softmax_scale)

    return y_float


class HLSLUTSoftmaxSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, logits_int_float, softmax_scale):
        logits_int = torch.round(logits_int_float).to(torch.int32)

        with torch.no_grad():
            y = hls_lut_softmax(
                logits_int,
                softmax_scale=softmax_scale,
            )

        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, grad_out):
        (y,) = ctx.saved_tensors

        # Standard softmax Jacobian-vector product:
        # dL/dx = y * (dL/dy - sum_j dL/dy_j * y_j)
        dot = (grad_out * y).sum(dim=-1, keepdim=True)
        grad_logits = y * (grad_out - dot)

        return grad_logits, None

def quant_tensor_to_int_ste(x, bit_width=8, signed=True):
    if hasattr(x, "value") and hasattr(x, "scale"):
        value = x.value
        scale = x.scale

        if hasattr(x, "zero_point") and x.zero_point is not None:
            zero_point = x.zero_point
        else:
            zero_point = 0.0

        x_float = value / scale + zero_point
    else:
        x_float = x

    if signed:
        qmin = -(2 ** (bit_width - 1))
        qmax = 2 ** (bit_width - 1) - 1
    else:
        qmin = 0
        qmax = 2 ** bit_width - 1

    x_q = torch.clamp(torch.round(x_float), qmin, qmax)

    # Straight-through estimator:
    # forward: x_q
    # backward: gradient as if identity
    return x_float + (x_q - x_float).detach()

# ---------------------------------------------------------------------
# Quantized Multi-Head Self-Attention
# ---------------------------------------------------------------------

class QuantMultiHeadSelfAttentionLayer(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.embed_dim = config["embed_dim"]
        self.num_heads = config["num_heads"]
        self.dropout = config.get("dropout", 0.0)

        assert self.embed_dim % self.num_heads == 0

        self.head_dim = self.embed_dim // self.num_heads
        self.scale = self.head_dim ** -0.5

        self.delta_tracking = True

        # ------------------------------------------------------------
        # Quantized Q, K, V projections
        # ------------------------------------------------------------
        self.q_proj = QuantLinear(
            self.embed_dim,
            self.embed_dim,
            bias=True,
            input_quant=Int8ActPerTensorFloat,
            weight_quant=Int8WeightPerTensorFloat,
            bias_quant=Int32Bias,
            output_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True,
        )

        self.k_proj = QuantLinear(
            self.embed_dim,
            self.embed_dim,
            bias=True,
            input_quant=self.q_proj.input_quant,
            weight_quant=Int8WeightPerTensorFloat,
            bias_quant=Int32Bias,
            output_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True,
        )

        self.v_proj = QuantLinear(
            self.embed_dim,
            self.embed_dim,
            bias=True,
            input_quant=self.q_proj.input_quant,
            weight_quant=Int8WeightPerTensorFloat,
            bias_quant=Int32Bias,
            output_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True,
        )

        # Quantize QK^T logits before the LUT softmax.
        self.logit_quant = QuantIdentity(
            act_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True,
        )

        # Quantize softmax output before A @ V.
        self.soft_quant = QuantIdentity(
            act_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True,
        )

        self.drop = nn.Dropout(self.dropout)

        # ------------------------------------------------------------
        # Output projection
        # ------------------------------------------------------------
        self.out_proj = QuantLinear(
            self.embed_dim,
            self.embed_dim,
            bias=True,
            input_quant=Int8ActPerTensorFloat,
            weight_quant=Int8WeightPerTensorFloat,
            bias_quant=Int32Bias,
            output_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True,
        )

    def forward(self, x):
        """
        Input:
            x: [B, N, E]

        Output:
            y: [B, N, E]
        """

        B, N, E = x.shape
        assert E == self.embed_dim

        # ------------------------------------------------------------
        # 1. Quantized projections
        # ------------------------------------------------------------
        q_qt = self.q_proj(x)
        k_qt = self.k_proj(x)
        v_qt = self.v_proj(x)

        q = _value(q_qt)
        k = _value(k_qt)
        v = _value(v_qt)

        # ------------------------------------------------------------
        # 2. Split heads
        #
        # [B, N, E] -> [B, H, N, head_dim]
        # ------------------------------------------------------------
        q = q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        # ------------------------------------------------------------
        # 3. Attention logits
        #
        # attn_logits: [B, H, N, N]
        # ------------------------------------------------------------
        

        # ------------------------------------------------------------
        attn_logits = torch.matmul(q, k.transpose(-2, -1))

        # Scale by 1 / sqrt(head_dim)
        attn_logits = attn_logits * self.scale

        attn_logits_qt = self.logit_quant(attn_logits)

        attn_logits_int_ste = quant_tensor_to_int_ste(
            attn_logits_qt,
            bit_width=8,
            signed=True,
        )

        attn = HLSLUTSoftmaxSTE.apply(
            attn_logits_int_ste,
            127,
        )

        # Quantize logits before softmax
        """
        attn_logits_qt = self.logit_quant(attn_logits)

        # Convert quantized logits to integer domain,
        # because your HLS LUT softmax operates on integers.
        attn_logits_int = _quant_tensor_to_int(
            attn_logits_qt,
            bit_width=8,
            signed=True,
        )

        # ------------------------------------------------------------
        # 4. HLS-like LUT softmax
        # ------------------------------------------------------------
        attn = hls_lut_softmax(
            attn_logits_int,
            softmax_scale=127,
        )
        """
        # Optional quantization after softmax
        attn_qt = self.soft_quant(attn)
        attn = _value(attn_qt)

        attn = self.drop(attn)

        # ------------------------------------------------------------
        # 5. A @ V
        #
        # attn: [B, H, N, N]
        # v:    [B, H, N, head_dim]
        # p:    [B, H, N, head_dim]
        # ------------------------------------------------------------
        p = torch.matmul(attn, v)

        # ------------------------------------------------------------
        # 6. Merge heads
        #
        # [B, H, N, head_dim] -> [B, N, E]
        # ------------------------------------------------------------
        p = p.transpose(1, 2).contiguous().view(B, N, E)

        # ------------------------------------------------------------
        # 7. Output projection
        # ------------------------------------------------------------
        y = self.out_proj(p)

        return y

In [13]:
class QuantBlockBatch(nn.Module):
    def __init__(self, config):
        super().__init__()
        D = config["embed_dim"]
        self.mha = QuantMultiHeadSelfAttentionLayer(config)
        self.mlp = QuantMLP(config)
        self.bn1 = nn.BatchNorm1d(D)
        self.bn2 = nn.BatchNorm1d(D)

        self.requant_mha = QuantIdentity(act_quant = self.mha.out_proj.output_quant,
                            return_quant_tensor=True)
        self.requant_mlp = QuantIdentity(act_quant = self.mlp.fc2.output_quant,
                            return_quant_tensor=True)

        self.quant_bn1_out = QuantIdentity(
            act_quant=self.mha.q_proj.input_quant, return_quant_tensor=True
        )
        self.quant_bn2_out = QuantIdentity(
            act_quant=self.mlp.fc1.input_quant, return_quant_tensor=True
        )

        self.quant_res0 = QuantIdentity(
            act_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True,
        )

        self.quant_res1 = QuantIdentity(
            act_quant=Int8ActPerTensorFloat,
            return_quant_tensor=True,
        )

    def _bn(self, bn, x):              # x: [B, N, D]
        x = x.transpose(1, 2)          # [B, D, N]
        x = bn(x)                      # BN over features
        return x.transpose(1, 2)       # back to [B, N, D]

    def forward(self, x):
        y_in = self.quant_bn1_out(self._bn(self.bn1, x))
        y = self.mha(y_in)
        x = self.quant_res0(self.requant_mha(x) + y)

        z_in = self.quant_bn2_out(self._bn(self.bn2, x))
        z = self.mlp(z_in)
        return self.quant_res1(self.requant_mlp(x) + z)

In [14]:
class ViT(nn.Module):
    def __init__(self,config):
        super().__init__()
        print(config)
        embed_dim = config["embed_dim"]
        depth = config["depth"]
        num_classes = config["num_classes"]

        self.patch = QuantPatchEmbeddings(config)
        #self.norm = nn.LayerNorm(embed_dim)
        self.norm = nn.BatchNorm1d(embed_dim)
        self.blocks = nn.ModuleList([
            #QuantBlock(config)
            QuantBlockBatch(config)
            for _ in range(depth)
        ])

        self.head = QuantLinear(
            embed_dim, 
            num_classes,
            bias=True,
            input_quant= self.blocks[-1].quant_bn2_out.act_quant,
            weight_quant = Int8WeightPerTensorFloat,
            bias_quant=Int8Bias,
            output_quant=Int8ActPerTensorFloat            
        )

        
    def _bn(self, bn, x):              # x: [B, N, D]
        x = x.transpose(1, 2)          # [B, D, N]
        x = bn(x)                      # BN over features, if not transpose we bn over tokens
        return x.transpose(1, 2) 

    def forward(self,x):

        x = self.patch(x)  # (B, N, D)
        for block in self.blocks:
            x = block(x)
        x = self._bn(self.norm,x)
        cls_token = x[:, 0]  # Use CLS token
        return self.head(cls_token)



        

In [15]:

model = ViT(config).to(device)

EPOCHS = config["epochs"]

EPOCHS = 10

vit_optim_train = False

if vit_optim_train:
      criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

      optimizer = torch.optim.AdamW(
      model.parameters(),
      lr=1e-4,
      weight_decay=5e-2
      )

      scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",       # because we monitor val_loss
            factor=0.5,       # new_lr = old_lr * 0.5
            patience=5,       # wait 5 bad epochs before reducing
            threshold=1e-4,
            min_lr=1e-6
            )
else:
      criterion = nn.CrossEntropyLoss()
      optimizer = optim.Adam(model.parameters(), lr=3e-4)

train_ = False

if train_:

      for epoch in range(EPOCHS):
            train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
            test_loss, test_acc = test(model, test_loader, criterion, device)

            if vit_optim_train:
                  scheduler.step(test_loss)
                  print(
                  f"[Epoch {epoch+1}/{EPOCHS}] "
                  f"LR: {scheduler.get_last_lr()[0]:.6f} | "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
                  f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}"
            )
            else:
                  print(f"[Epoch {epoch+1}/{EPOCHS}] "f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

      model.to("cpu")
      torch.save(model.state_dict(), "ViT_Quant_2.pth")
else:
      model.load_state_dict(torch.load("./ViT_Quant_2.pth"))
      model.eval()

{'batch_size': 32, 'patch_size': 8, 'mlp_size': 768, 'embed_dim': 192, 'num_heads': 4, 'dropout': 0.1, 'image_size': 32, 'num_classes': 10, 'num_channels': 1, 'qkv_bias': True, 'depth': 1, 'epochs': 10}


/home/fede/miniconda3/envs/ptorch/lib/python3.9/site-packages/brevitas/nn/mixin/base.py:59: UserWarning: Keyword arguments are being passed but they not being used.
  warn('Keyword arguments are being passed but they not being used.')
/tmp/ipykernel_882429/3009677371.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any u

In [16]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = total - trainable

    return {
        "total": total,
        "trainable": trainable,
        "non_trainable": non_trainable,
    }


stats = count_parameters(model)

print(f"Total parameters:       {stats['total']:,}")
print(f"Trainable parameters:   {stats['trainable']:,}")
print(f"Non-trainable params:   {stats['non_trainable']:,}")

Total parameters:       464,667
Trainable parameters:   464,667
Non-trainable params:   0


In [ ]:
import math
from pathlib import Path

import numpy as np
import torch


# ============================================================
# Fixed-point export format
# ============================================================

INT32_MIN = -(1 << 31)
INT32_MAX = (1 << 31) - 1

# Quantization scales are usually small numbers such as 0.0039.
# Q30 preserves them accurately as int32_t.
SCALE_FRAC_BITS = 30

# BN parameters, attention scale, and generic real-valued params.
# Q16 is safer for values that may be larger than 1.
PARAM_FRAC_BITS = 16


def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def flatten_c(x):
    return to_numpy(x).reshape(-1)


def fixed_to_int32_np(x, frac_bits, name=""):
    x = to_numpy(x).astype(np.float64)
    q = np.round(x * float(1 << frac_bits))

    if q.size > 0:
        q_min = q.min()
        q_max = q.max()

        if q_min < INT32_MIN or q_max > INT32_MAX:
            raise OverflowError(
                f"{name} overflows int32_t with frac_bits={frac_bits}: "
                f"min={q_min}, max={q_max}"
            )

    return q.astype(np.int32)


def c_array_str(name, arr, ctype, values_per_line=16):
    arr = flatten_c(arr)

    if ctype == "float":
        vals = [f"{float(v):.8f}f" for v in arr]
    elif ctype == "int8_t":
        vals = [str(int(np.int8(v))) for v in arr]
    elif ctype == "int32_t":
        vals = [str(int(np.int32(v))) for v in arr]
    else:
        raise ValueError(f"Unsupported ctype: {ctype}")

    lines = []
    for i in range(0, len(vals), values_per_line):
        lines.append("    " + ", ".join(vals[i:i + values_per_line]))

    return f"const {ctype} {name}[{len(arr)}] = {{\n" + ",\n".join(lines) + "\n};\n"


def c_scalar_str(name, value, ctype):
    if ctype == "float":
        return f"const {ctype} {name} = {float(value):.8f}f;\n"
    elif ctype == "int32_t":
        return f"const {ctype} {name} = {int(np.int32(value))};\n"
    else:
        raise ValueError(f"Unsupported scalar ctype: {ctype}")


def c_fixed_scalar_str(name, value, frac_bits):
    q = fixed_to_int32_np(np.asarray([value]), frac_bits, name=name)[0]
    return c_scalar_str(name, q, "int32_t")


def c_fixed_array_str(name, arr, frac_bits, values_per_line=16):
    q = fixed_to_int32_np(arr, frac_bits, name=name)
    return c_array_str(name, q, "int32_t", values_per_line=values_per_line)


def quantize_np_to_int8(x, scale, zp=0):
    q = np.round(x / scale).astype(np.int32) + int(zp)
    q = np.clip(q, -128, 127)
    return q.astype(np.int8)


def first_scalar(x):
    if x is None:
        return None

    if isinstance(x, torch.Tensor):
        if x.numel() == 0:
            return None
        return float(x.detach().cpu().reshape(-1)[0])

    if isinstance(x, np.ndarray):
        if x.size == 0:
            return None
        return float(x.reshape(-1)[0])

    return float(x)


def get_quant_scale_from_proxy(proxy):
    for attr in ["scale", "scaling_impl"]:
        if hasattr(proxy, attr):
            obj = getattr(proxy, attr)
            try:
                val = obj() if callable(obj) else obj
                s = first_scalar(val)
                if s is not None:
                    return s
            except Exception:
                pass

    try:
        tq = proxy.fused_activation_quant_proxy.tensor_quant
        if hasattr(tq, "scale"):
            s = first_scalar(tq.scale())
            if s is not None:
                return s
    except Exception:
        pass

    return None


def get_quant_zp_from_proxy(proxy, default=0):
    for attr in ["zero_point"]:
        if hasattr(proxy, attr):
            obj = getattr(proxy, attr)
            try:
                val = obj() if callable(obj) else obj
                zp = first_scalar(val)
                if zp is not None:
                    return int(round(zp))
            except Exception:
                pass

    try:
        tq = proxy.fused_activation_quant_proxy.tensor_quant
        if hasattr(tq, "zero_point"):
            zp = first_scalar(tq.zero_point())
            if zp is not None:
                return int(round(zp))
    except Exception:
        pass

    return int(default)

def qt_scale(x, name):
    if not hasattr(x, "scale") or x.scale is None:
        raise RuntimeError(f"Activation {name} has no .scale")

    s = first_scalar(x.scale)
    if s is None:
        raise RuntimeError(f"Could not read scale from activation {name}")

    return float(s)


def qt_zp(x, default=0):
    if hasattr(x, "zero_point") and x.zero_point is not None:
        zp = first_scalar(x.zero_point)
        if zp is not None:
            return int(round(zp))
    return int(default)


def collect_vit_activation_scales(model, sample_input=None, test_loader=None, device="cpu"):
    activations = {}

    targets = {
        "patch",
        "blocks.0.mha.q_proj",
        "blocks.0.mha.k_proj",
        "blocks.0.mha.v_proj",
        "blocks.0.mha.out_proj",
        "blocks.0.mha.logit_quant",
        "blocks.0.mha.soft_quant",
        "blocks.0.mlp.fc2",
        "blocks.0.quant_res0",
        "blocks.0.quant_res1",
    }

    def hook_fn(name):
        def hook(module, inputs, output):
            activations[name] = output
        return hook

    hooks = []
    for name, module in model.named_modules():
        if name in targets:
            hooks.append(module.register_forward_hook(hook_fn(name)))

    was_training = model.training

    try:
        model.to(device)
        model.eval()

        if sample_input is None:
            if test_loader is None:
                raise RuntimeError(
                    "Need sample_input or test_loader to collect activation scales."
                )

            batch = next(iter(test_loader))
            images = batch[0] if isinstance(batch, (tuple, list)) else batch
            sample_input = images[:1]

        sample_input = sample_input.to(device)

        with torch.no_grad():
            _ = model(sample_input)

    finally:
        for h in hooks:
            h.remove()

        if was_training:
            model.train()

    missing = sorted(targets - set(activations.keys()))
    if missing:
        raise RuntimeError(f"Missing activation hooks for: {missing}")

    return {
        "patch_scale": qt_scale(activations["patch"], "patch"),
        "q_scale": qt_scale(activations["blocks.0.mha.q_proj"], "q_proj"),
        "k_scale": qt_scale(activations["blocks.0.mha.k_proj"], "k_proj"),
        "v_scale": qt_scale(activations["blocks.0.mha.v_proj"], "v_proj"),
        "out_proj_scale": qt_scale(activations["blocks.0.mha.out_proj"], "out_proj"),
        "logit_scale": qt_scale(activations["blocks.0.mha.logit_quant"], "logit_quant"),
        "logit_zp": qt_zp(activations["blocks.0.mha.logit_quant"], 0),
        "soft_scale": qt_scale(activations["blocks.0.mha.soft_quant"], "soft_quant"),
        "soft_zp": qt_zp(activations["blocks.0.mha.soft_quant"], 0),
        "fc2_scale": qt_scale(activations["blocks.0.mlp.fc2"], "fc2"),
        "res0_scale": qt_scale(activations["blocks.0.quant_res0"], "quant_res0"),
        "res1_scale": qt_scale(activations["blocks.0.quant_res1"], "quant_res1"),
    }

def get_bn_params(bn):
    rm = to_numpy(bn.running_mean).astype(np.float32)
    rv = to_numpy(bn.running_var).astype(np.float32)
    gamma = to_numpy(bn.weight).astype(np.float32)
    beta = to_numpy(bn.bias).astype(np.float32)
    eps = float(bn.eps)

    rs = 1.0 / np.sqrt(rv + eps)

    return {
        "rm": rm,
        "rs": rs.astype(np.float32),
        "scale": gamma,
        "bias": beta,
    }


def ratio_to_mult_shift(acc_scale, target_scale, shift=16):
    """
    MULT / 2^SHIFT ~= acc_scale / target_scale
    """
    ratio = float(acc_scale) / float(target_scale)
    mult = int(round(ratio * (1 << shift)))
    return mult, shift


def get_quantlinear_export(qlinear, input_act_scale):
    qw = qlinear.quant_weight()

    if not hasattr(qw, "value") or not hasattr(qw, "scale"):
        raise RuntimeError(f"quant_weight() did not return a QuantTensor for {qlinear}")

    w_real = qw.value.detach().cpu()
    w_scale = float(qw.scale.detach().cpu().reshape(-1)[0])

    w_int = torch.round(w_real / w_scale)
    w_int = torch.clamp(w_int, -128, 127).to(torch.int8).numpy()

    # Export linear weights as [in_features, out_features]
    # because PyTorch stores them as [out_features, in_features].
    w_int = w_int.T

    if qlinear.bias is not None:
        bias_scale = float(input_act_scale) * float(w_scale)
        b_real = qlinear.bias.detach().cpu()
        b_int = torch.round(b_real / bias_scale).to(torch.int32).numpy()
    else:
        b_int = np.zeros(qlinear.out_features, dtype=np.int32)

    return w_int, b_int, w_scale, 0


def get_quantconv_export(qconv, input_act_scale):
    qw = qconv.quant_weight()

    if not hasattr(qw, "value") or not hasattr(qw, "scale"):
        raise RuntimeError(f"quant_weight() did not return a QuantTensor for {qconv}")

    w_real = qw.value.detach().cpu()
    w_scale = float(qw.scale.detach().cpu().reshape(-1)[0])

    w_int = torch.round(w_real / w_scale)
    w_int = torch.clamp(w_int, -128, 127).to(torch.int8).numpy()

    if qconv.bias is not None:
        bias_scale = float(input_act_scale) * float(w_scale)
        b_real = qconv.bias.detach().cpu()
        b_int = torch.round(b_real / bias_scale).to(torch.int32).numpy()
    else:
        b_int = np.zeros(qconv.out_channels, dtype=np.int32)

    return w_int, b_int, w_scale, 0


def inspect_exported_weight(name, w_int):
    w_flat = w_int.reshape(-1)
    print(name)
    print("  shape:", w_int.shape)
    print("  min/max:", int(w_flat.min()), int(w_flat.max()))
    print("  nonzero count:", int(np.count_nonzero(w_flat)))
    print("  first 20:", w_flat[:20])


def inspect_exported_bias(name, b_int):
    b_flat = b_int.reshape(-1)
    print(name)
    print("  shape:", b_int.shape)
    print("  min/max:", int(b_flat.min()), int(b_flat.max()))
    print("  nonzero count:", int(np.count_nonzero(b_flat)))
    print("  first 20:", b_flat[:20])

def export_bn_affine_int(
    bn,
    input_scale,
    output_scale,
    output_zp=0,
    input_zp=0,
    shift=16,
):
    rm = to_numpy(bn.running_mean).astype(np.float64)
    rv = to_numpy(bn.running_var).astype(np.float64)
    gamma = to_numpy(bn.weight).astype(np.float64)
    beta = to_numpy(bn.bias).astype(np.float64)
    eps = float(bn.eps)

    inv_std = 1.0 / np.sqrt(rv + eps)

    # y_real = gamma * inv_std * (S_in * (q - zp_in) - rm) + beta
    # q_out = y_real / S_out + zp_out
    alpha = float(input_scale) * gamma * inv_std / float(output_scale)

    beta_q = (
        beta
        - rm * gamma * inv_std
        - float(input_zp) * float(input_scale) * gamma * inv_std
    ) / float(output_scale) + float(output_zp)

    mul = np.round(alpha * (1 << shift)).astype(np.int64)
    add = np.round(beta_q * (1 << shift)).astype(np.int64)

    if mul.min() < INT32_MIN or mul.max() > INT32_MAX:
        raise OverflowError("BN mul overflows int32_t")

    if add.min() < INT32_MIN or add.max() > INT32_MAX:
        raise OverflowError("BN add overflows int32_t")

    return mul.astype(np.int32), add.astype(np.int32), shift

def int32_literal(value, name=""):
    v = int(value)
    if v < INT32_MIN or v > INT32_MAX:
        raise OverflowError(f"{name} overflows int32_t: {v}")
    return str(v)


def c_define_int_str(name, value):
    return f"#define {name} {int32_literal(value, name)}\n"


def c_fixed_define_int_str(name, value, frac_bits):
    q = fixed_to_int32_np(np.asarray([value]), frac_bits, name=name)[0]
    return c_define_int_str(name, q)

def export_vit_to_c(
    model,
    config,
    out_dir_h="./vit_c_export",
    out_dir_c="./vit_c_export",
    verbose=True,
    test_loader=None,
    sample_input=None,
    device="cpu",
):
    model.eval()

    
    out_dir_h = Path(out_dir_h)
    out_dir_h.mkdir(parents=True, exist_ok=True)

    out_dir_c = Path(out_dir_c)
    out_dir_c.mkdir(parents=True, exist_ok=True)

    # ---------------- model paths ----------------
    patch = model.patch
    block = model.blocks[0]
    mha = block.mha
    mlp = block.mlp
    head = model.head

    # ---------------- config ----------------
    in_channels = int(config["num_channels"])
    height = int(config["image_size"])
    width = int(config["image_size"])
    patch_size = int(config["patch_size"])
    embed_dim = int(config["embed_dim"])
    num_heads = int(config["num_heads"])
    num_classes = int(config["num_classes"])

    seq_len = (height // patch_size) * (width // patch_size)
    attn_scale = math.sqrt(embed_dim // num_heads)

    # ---------------- activation scales / zero-points ----------------
        # Override critical scales using actual QuantTensor activations.
    seq_len = (height // patch_size) * (width // patch_size)
    attn_scale = math.sqrt(embed_dim // num_heads)

    act_scales = collect_vit_activation_scales(
        model,
        sample_input=sample_input,
        test_loader=test_loader,
        device=device,
    )

    head_dim = embed_dim // num_heads

    patch_out_scale = act_scales["patch_scale"]
    q_out_scale = act_scales["q_scale"]
    k_out_scale = act_scales["k_scale"]
    v_out_scale = act_scales["v_scale"]
    out_proj_out_scale = act_scales["out_proj_scale"]
    soft_out_scale = act_scales["soft_scale"]
    soft_out_zp = act_scales["soft_zp"]
    fc2_out_scale = act_scales["fc2_scale"]

    logit_out_scale = act_scales["logit_scale"]
    logit_out_zp = act_scales["logit_zp"]

    res_0_scale = act_scales["res0_scale"]
    res_1_scale = act_scales["res1_scale"]

    RATIO_SHIFT = 16
    RATIO1_SHIFT = 16
    OUT_RES_0_SHIFT = 16
    OUT_RES_1_SHIFT = 16

    INPUT_SCALE = get_quant_scale_from_proxy(patch.proj.input_quant)
    INPUT_ZERO_PT = get_quant_zp_from_proxy(patch.proj.input_quant, 0)

    #patch_out_scale = get_quant_scale_from_proxy(patch.proj.output_quant)
    patch_out_zp = get_quant_zp_from_proxy(patch.proj.output_quant, 0)

    patch_weight_scale = get_quant_scale_from_proxy(patch.proj.weight_quant)

    bn1_out_scale = get_quant_scale_from_proxy(mha.q_proj.input_quant)
    bn1_out_zp = get_quant_zp_from_proxy(mha.q_proj.input_quant, 0)

    bn2_out_scale = get_quant_scale_from_proxy(mlp.fc1.input_quant)
    bn2_out_zp = get_quant_zp_from_proxy(mlp.fc1.input_quant, 0)

    #q_out_scale = get_quant_scale_from_proxy(mha.q_proj.output_quant)
    q_out_zp = get_quant_zp_from_proxy(mha.q_proj.output_quant, 0)

    #k_out_scale = get_quant_scale_from_proxy(mha.k_proj.output_quant)
    k_out_zp = get_quant_zp_from_proxy(mha.k_proj.output_quant, 0)

    #v_out_scale = get_quant_scale_from_proxy(mha.v_proj.output_quant)
    v_out_zp = get_quant_zp_from_proxy(mha.v_proj.output_quant, 0)

    out_proj_in_scale = get_quant_scale_from_proxy(mha.out_proj.input_quant)
    out_proj_in_zp = get_quant_zp_from_proxy(mha.out_proj.input_quant, 0)

    #out_proj_out_scale = get_quant_scale_from_proxy(mha.out_proj.output_quant)
    out_proj_out_zp = get_quant_zp_from_proxy(mha.out_proj.output_quant, 0)

    fc1_out_scale = get_quant_scale_from_proxy(mlp.fc1.output_quant)
    fc1_out_zp = get_quant_zp_from_proxy(mlp.fc1.output_quant, 0)

    fc2_in_scale = get_quant_scale_from_proxy(mlp.fc2.input_quant)
    fc2_in_zp = get_quant_zp_from_proxy(mlp.fc2.input_quant, 0)

    #fc2_out_scale = get_quant_scale_from_proxy(mlp.fc2.output_quant)
    fc2_out_zp = get_quant_zp_from_proxy(mlp.fc2.output_quant, 0)

    S_out_proj_in = get_quant_scale_from_proxy(model.blocks[0].mha.out_proj.input_quant)
    print("S_out_proh_in:", S_out_proj_in)

    """
    soft_out_scale = None
    soft_out_zp = 0

    if hasattr(mha, "soft_quant") and hasattr(mha.soft_quant, "act_quant"):
        soft_out_scale = get_quant_scale_from_proxy(mha.soft_quant.act_quant)
        soft_out_zp = get_quant_zp_from_proxy(mha.soft_quant.act_quant, 0)

    if soft_out_scale is None:
        soft_out_scale = 1.0 / 255.0
        soft_out_zp = 0
    """

    # Residual output domains
    res1_scale = out_proj_out_scale
    res1_zp = out_proj_out_zp

    res2_scale = fc2_out_scale
    res2_zp = fc2_out_zp

    # Classifier input domain
    head_in_scale = get_quant_scale_from_proxy(head.input_quant)
    head_in_zp = get_quant_zp_from_proxy(head.input_quant, 0)

    debug_scales = {
        "INPUT_SCALE": INPUT_SCALE,
        "patch_out_scale": patch_out_scale,
        "patch_weight_scale": patch_weight_scale,
        "bn1_out_scale": bn1_out_scale,
        "bn2_out_scale": bn2_out_scale,
        "q_out_scale": q_out_scale,
        "k_out_scale": k_out_scale,
        "v_out_scale": v_out_scale,
        "soft_out_scale": soft_out_scale,
        "out_proj_in_scale": out_proj_in_scale,
        "out_proj_out_scale": out_proj_out_scale,
        "fc1_out_scale": fc1_out_scale,
        "fc2_in_scale": fc2_in_scale,
        "fc2_out_scale": fc2_out_scale,
        "head_in_scale": head_in_scale,
    }

    if verbose:
        print("=== Export scales ===")
        for k, v in debug_scales.items():
            print(k, v)

    missing = [k for k, v in debug_scales.items() if v is None]
    if missing:
        raise RuntimeError(f"Missing quant scales for: {missing}")

    # ---------------- weights / biases ----------------
    patch_w, patch_b, patch_w_scale, _ = get_quantconv_export(patch.proj, INPUT_SCALE)

    w_q_0, bias_q_0, wq_scale, _ = get_quantlinear_export(mha.q_proj, bn1_out_scale)
    w_k_0, bias_k_0, wk_scale, _ = get_quantlinear_export(mha.k_proj, bn1_out_scale)
    w_v_0, bias_v_0, wv_scale, _ = get_quantlinear_export(mha.v_proj, bn1_out_scale)
    head_0, bias_h_0, wh_scale, _ = get_quantlinear_export(mha.out_proj, out_proj_in_scale)

    w_1_0, bias_1_0, w1_scale, _ = get_quantlinear_export(mlp.fc1, bn2_out_scale)
    w_2_0, bias_2_0, w2_scale, _ = get_quantlinear_export(mlp.fc2, fc2_in_scale)

    clas_weights_0, clas_bias_0, head_w_scale, _ = get_quantlinear_export(head, head_in_scale)

    # ---------------- cls token / positional encoding ----------------
    cls_token = quantize_np_to_int8(
        to_numpy(patch.cls_token_param),
        patch_out_scale,
        patch_out_zp,
    )

    pos_enc = quantize_np_to_int8(
        to_numpy(patch.pos_embed_param),
        patch_out_scale,
        patch_out_zp,
    )

    # ---------------- BN params ----------------
    bn1 = get_bn_params(block.bn1)
    bn2 = get_bn_params(block.bn2)
    bn_final = get_bn_params(model.norm)

    # ---------------- requant params ----------------
    patch_acc_scale = float(INPUT_SCALE) * float(patch_w_scale)
    PATCH_MULT, PATCH_SHIFT = ratio_to_mult_shift(
        patch_acc_scale,
        patch_out_scale,
        shift=16,
    )
    PATCH_ZERO_PT = int(patch_out_zp)

    MULT_q, SHIFT_q = ratio_to_mult_shift(
        float(bn1_out_scale) * float(wq_scale),
        q_out_scale,
        shift=16,
    )
    ZP_OUT_q = int(q_out_zp)

    MULT_k, SHIFT_k = ratio_to_mult_shift(
        float(bn1_out_scale) * float(wk_scale),
        k_out_scale,
        shift=16,
    )
    ZP_OUT_k = int(k_out_zp)

    MULT_vproj, SHIFT_vproj = ratio_to_mult_shift(
        float(bn1_out_scale) * float(wv_scale),
        v_out_scale,
        shift=16,
    )
    ZP_OUT_vproj = int(v_out_zp)

    # Attention score tensor A = Q @ K^T.
    # qk accumulator scale = q_out_scale * k_out_scale.
    SCALE_x = float(q_out_scale) * float(k_out_scale)
    ZERO_x = 0

    MULT_a, SHIFT_a = ratio_to_mult_shift(
        float(q_out_scale) * float(k_out_scale),
        SCALE_x,
        shift=16,
    )
    ZP_OUT_a = 0

    # Softmax output domain.
    SCALE_x1 = float(soft_out_scale)
    ZERO_x1 = int(soft_out_zp)

    # A @ V -> out_proj input domain.
    MULT_v, SHIFT_v = ratio_to_mult_shift(
        float(SCALE_x1) * float(v_out_scale),
        out_proj_in_scale,
        shift=16,
    )
    ZP_OUT_v = int(out_proj_in_zp)

    # out_proj -> residual add domain.
    MULT_h, SHIFT_h = ratio_to_mult_shift(
        float(out_proj_in_scale) * float(wh_scale),
        out_proj_out_scale,
        shift=16,
    )
    ZP_OUT_h = int(out_proj_out_zp)

    # fc1 -> fc2 input domain.
    MULT_fc1, SHIFT_fc1 = ratio_to_mult_shift(
        float(bn2_out_scale) * float(w1_scale),
        fc2_in_scale,
        shift=16,
    )
    ZP_OUT_fc1 = int(fc2_in_zp)

    # fc2 -> residual add domain.
    MULT_fc2, SHIFT_fc2 = ratio_to_mult_shift(
        float(fc2_in_scale) * float(w2_scale),
        fc2_out_scale,
        shift=16,
    )
    ZP_OUT_fc2 = int(fc2_out_zp)

    # Residual skip requants.
    RATIO, RATIO_SHIFT = ratio_to_mult_shift(
        patch_out_scale,
        res1_scale,
        shift=16,
    )

    RATIO1, RATIO1_SHIFT = ratio_to_mult_shift(
        res1_scale,
        res2_scale,
        shift=16,
    )    # Attention logits:
    # S = Q @ K^T / sqrt(head_dim)
    #
    # Accumulator real scale:
    #     q_out_scale * k_out_scale * head_dim^(-0.5)
    #
    # Target quantized logit scale:
    #     logit_out_scale
    SCALE_x = float(logit_out_scale)
    ZERO_x = int(logit_out_zp)

    MULT_a = int(
        (
            float(q_out_scale)
            * float(k_out_scale)
            * (float(head_dim) ** -0.5)
            / float(logit_out_scale)
        )
        * (1 << 16)
    )
    SHIFT_a = 16
    ZP_OUT_a = int(logit_out_zp)

    # Softmax output domain.
    SCALE_x1 = float(soft_out_scale)
    ZERO_x1 = int(soft_out_zp)

    # Converts softmax real values into the quantized softmax domain.
    MULT_soft = int(round((1.0 / (127.0 * float(soft_out_scale))) * (1 << 16)))

    # Softmax @ V -> out_proj input domain.
    #
    # Important: denominator is out_proj INPUT scale, not out_proj output scale.
    MULT_kv = int(
        round(
            (
                float(soft_out_scale)
                * float(v_out_scale)
                / float(out_proj_in_scale)
            )
            * (1 << 16)
        )
    )

    # Keep old name as alias if your HLS still uses MULT_v.
    MULT_v = MULT_kv
    SHIFT_v = 16
    ZP_OUT_v = int(out_proj_in_zp)

    # out_proj -> first residual add domain.
    MULT_h, SHIFT_h = ratio_to_mult_shift(
        float(out_proj_in_scale) * float(wh_scale),
        out_proj_out_scale,
        shift=16,
    )
    ZP_OUT_h = int(out_proj_out_zp)

    # First residual:
    #   patch branch -> out_proj scale before add
    #   add output -> quant_res0 scale
    RATIO = int(round((float(patch_out_scale) / float(out_proj_out_scale)) * (1 << 16)))

    OUT_RES_0 = int((float(out_proj_out_scale) / float(res_0_scale)) * (1 << 16))

    # fc1 -> fc2 input domain.
    MULT_fc1, SHIFT_fc1 = ratio_to_mult_shift(
        float(bn2_out_scale) * float(w1_scale),
        fc2_in_scale,
        shift=16,
    )
    ZP_OUT_fc1 = int(fc2_in_zp)

    # fc2 output domain.
    MULT_fc2, SHIFT_fc2 = ratio_to_mult_shift(
        float(fc2_in_scale) * float(w2_scale),
        fc2_out_scale,
        shift=16,
    )
    ZP_OUT_fc2 = int(fc2_out_zp)

    # Second residual:
    #   quant_res0 branch -> fc2 output scale before add
    #   add output -> quant_res1 scale
    RATIO1 = int(round((float(res_0_scale) / float(fc2_out_scale)) * (1 << 16)))

    OUT_RES_1 = int(round((float(fc2_out_scale) / float(res_1_scale)) * (1 << 16)))

    # BN / block scales used by the C path.

    BN_SCALE = patch_out_scale
    BN0_OUT_SCALE = bn1_out_scale
    BN1_SCALE = res_0_scale
    MLP_SCALE = bn2_out_scale
    RES2_SCALE = res_1_scale

    # Classifier output stays raw int32 logits in current C path.
    HEAD_MULT = 1
    HEAD_SHIFT = 0
    HEAD_ZERO_PT = 0

    bn0_mul, bn0_add, BN0_SHIFT = export_bn_affine_int(
        block.bn1,
        input_scale=BN_SCALE,
        output_scale=BN0_OUT_SCALE,
        output_zp=bn1_out_zp,
        input_zp=0,
        shift=16,
    )
    bn1_mul, bn1_add, BN1_SHIFT = export_bn_affine_int(
    block.bn2,
    input_scale=BN1_SCALE,
    output_scale=MLP_SCALE,
    output_zp=bn2_out_zp,
    input_zp=0,
    shift=16,
    )
    

    print(f"BN0 MULT: {bn0_mul}, BN0_ADD: {bn0_add}, BN0_SHIFT: {BN0_SHIFT}")
    print(f"BN1 MULT: {bn1_mul}, BN1_ADD: {bn1_add}, BN1_SHIFT: {BN1_SHIFT}")


    if verbose:
        print("\n=== Patch requant ===")
        print("patch_acc_scale =", patch_acc_scale)
        print("PATCH_MULT =", PATCH_MULT)
        print("PATCH_SHIFT =", PATCH_SHIFT)
        print("PATCH_ZERO_PT =", PATCH_ZERO_PT)

        print("\n=== Projection requants ===")
        print("MULT_q =", MULT_q, "SHIFT_q =", SHIFT_q, "ZP_OUT_q =", ZP_OUT_q)
        print("MULT_k =", MULT_k, "SHIFT_k =", SHIFT_k, "ZP_OUT_k =", ZP_OUT_k)
        print(
            "MULT_vproj =",
            MULT_vproj,
            "SHIFT_vproj =",
            SHIFT_vproj,
            "ZP_OUT_vproj =",
            ZP_OUT_vproj,
        )

        print("\n=== Attention requants ===")
        print("SCALE_x =", SCALE_x, "ZERO_x =", ZERO_x)
        print("MULT_a =", MULT_a, "SHIFT_a =", SHIFT_a, "ZP_OUT_a =", ZP_OUT_a)
        print("SCALE_x1 =", SCALE_x1, "ZERO_x1 =", ZERO_x1)
        print("MULT_v =", MULT_v, "SHIFT_v =", SHIFT_v, "ZP_OUT_v =", ZP_OUT_v)

        print("\n=== MLP / out proj requants ===")
        print("MULT_h =", MULT_h, "SHIFT_h =", SHIFT_h, "ZP_OUT_h =", ZP_OUT_h)
        print(
            "MULT_fc1 =",
            MULT_fc1,
            "SHIFT_fc1 =",
            SHIFT_fc1,
            "ZP_OUT_fc1 =",
            ZP_OUT_fc1,
        )
        print(
            "MULT_fc2 =",
            MULT_fc2,
            "SHIFT_fc2 =",
            SHIFT_fc2,
            "ZP_OUT_fc2 =",
            ZP_OUT_fc2,
        )

        print("\n=== New attention formulas ===")
        print("head_dim =", head_dim)
        print("q_out_scale =", q_out_scale)
        print("k_out_scale =", k_out_scale)
        print("logit_out_scale =", logit_out_scale)
        print("MULT_a =", MULT_a)
        print("soft_out_scale =", soft_out_scale)
        print("MULT_soft =", MULT_soft)
        print("v_out_scale =", v_out_scale)
        print("out_proj_in_scale =", out_proj_in_scale)
        print("MULT_kv =", MULT_kv)

        print("\n=== New residual formulas ===")
        print("patch_out_scale =", patch_out_scale)
        print("out_proj_out_scale =", out_proj_out_scale)
        print("res_0_scale =", res_0_scale)
        print("fc2_out_scale =", fc2_out_scale)
        print("res_1_scale =", res_1_scale)
        print("RATIO =", RATIO)
        print("OUT_RES_0 =", OUT_RES_0)
        print("RATIO1 =", RATIO1)
        print("OUT_RES_1 =", OUT_RES_1)

        print("\n=== Exported tensor sanity ===")
        inspect_exported_weight("patch_kernels_0", patch_w)
        inspect_exported_bias("patch_bias_0", patch_b)
        inspect_exported_weight("w_q_0", w_q_0)
        inspect_exported_bias("bias_q_0", bias_q_0)
        inspect_exported_weight("w_1_0", w_1_0)
        inspect_exported_bias("bias_1_0", bias_1_0)
        inspect_exported_weight("clas_weights_0", clas_weights_0)
        inspect_exported_bias("clas_bias_0", clas_bias_0)

    # ---------------- write vit_params.h ----------------

    scalar_defines = []

    # Compile-time model dimensions.
    scalar_defines.append(c_define_int_str("SCALE_FRAC_BITS", SCALE_FRAC_BITS))
    scalar_defines.append(c_define_int_str("PARAM_FRAC_BITS", PARAM_FRAC_BITS))
    scalar_defines.append("\n")

    scalar_defines.append(c_define_int_str("IN_CHANNELS", in_channels))
    scalar_defines.append(c_define_int_str("HEIGHT", height))
    scalar_defines.append(c_define_int_str("WIDTH", width))
    scalar_defines.append(c_define_int_str("INPUT_DIM", in_channels * height * width))
    scalar_defines.append(c_define_int_str("PATCH", patch_size))
    scalar_defines.append(c_define_int_str("SEQUENCE_LENGTH", seq_len))
    scalar_defines.append(c_define_int_str("EMBED_DIM", embed_dim))
    scalar_defines.append(c_define_int_str("T", seq_len))
    scalar_defines.append(c_define_int_str("E", embed_dim))
    scalar_defines.append(c_define_int_str("F", embed_dim * 4))
    scalar_defines.append(c_define_int_str("H", num_heads))
    scalar_defines.append(c_define_int_str("OUT_CLS", num_classes))
    scalar_defines.append("\n")

    # Scales exported as Q30 int32_t.
    scalar_defines.append(c_fixed_define_int_str("INPUT_SCALE", INPUT_SCALE, SCALE_FRAC_BITS))
    scalar_defines.append(c_define_int_str("INPUT_ZERO_PT", INPUT_ZERO_PT))
    scalar_defines.append("\n")

    scalar_defines.append(c_fixed_define_int_str("BN_SCALE", BN_SCALE, SCALE_FRAC_BITS))
    scalar_defines.append(c_fixed_define_int_str("BN0_OUT_SCALE", BN0_OUT_SCALE, SCALE_FRAC_BITS))
    scalar_defines.append(c_fixed_define_int_str("BN1_SCALE", BN1_SCALE, SCALE_FRAC_BITS))
    scalar_defines.append(c_fixed_define_int_str("MLP_SCALE", MLP_SCALE, SCALE_FRAC_BITS))
    scalar_defines.append(c_fixed_define_int_str("RES2_SCALE", RES2_SCALE, SCALE_FRAC_BITS))
    scalar_defines.append("\n")

    scalar_defines.append(c_fixed_define_int_str("SCALE_x", SCALE_x, SCALE_FRAC_BITS))
    scalar_defines.append(c_define_int_str("ZERO_x", ZERO_x))
    scalar_defines.append(c_fixed_define_int_str("SCALE_x1", SCALE_x1, SCALE_FRAC_BITS))
    scalar_defines.append(c_define_int_str("ZERO_x1", ZERO_x1))
    scalar_defines.append("\n")

    # Attention scale exported as Q16 int32_t.
    scalar_defines.append(c_fixed_define_int_str("ATTN_SCALE", attn_scale, PARAM_FRAC_BITS))
    scalar_defines.append("\n")

    scalar_defines.append(c_define_int_str("PATCH_MULT", PATCH_MULT))
    scalar_defines.append(c_define_int_str("PATCH_SHIFT", PATCH_SHIFT))
    scalar_defines.append(c_define_int_str("PATCH_ZERO_PT", PATCH_ZERO_PT))
    scalar_defines.append("\n")

    scalar_defines.append(c_define_int_str("MULT_q", MULT_q))
    scalar_defines.append(c_define_int_str("SHIFT_q", SHIFT_q))
    scalar_defines.append(c_define_int_str("ZP_OUT_q", ZP_OUT_q))

    scalar_defines.append(c_define_int_str("MULT_k", MULT_k))
    scalar_defines.append(c_define_int_str("SHIFT_k", SHIFT_k))
    scalar_defines.append(c_define_int_str("ZP_OUT_k", ZP_OUT_k))

    scalar_defines.append(c_define_int_str("MULT_vproj", MULT_vproj))
    scalar_defines.append(c_define_int_str("SHIFT_vproj", SHIFT_vproj))
    scalar_defines.append(c_define_int_str("ZP_OUT_vproj", ZP_OUT_vproj))
    scalar_defines.append("\n")

    scalar_defines.append(c_define_int_str("MULT_h", MULT_h))
    scalar_defines.append(c_define_int_str("SHIFT_h", SHIFT_h))
    scalar_defines.append(c_define_int_str("ZP_OUT_h", ZP_OUT_h))

    scalar_defines.append(c_define_int_str("MULT_fc1", MULT_fc1))
    scalar_defines.append(c_define_int_str("SHIFT_fc1", SHIFT_fc1))
    scalar_defines.append(c_define_int_str("ZP_OUT_fc1", ZP_OUT_fc1))

    scalar_defines.append(c_define_int_str("MULT_fc2", MULT_fc2))
    scalar_defines.append(c_define_int_str("SHIFT_fc2", SHIFT_fc2))
    scalar_defines.append(c_define_int_str("ZP_OUT_fc2", ZP_OUT_fc2))
    scalar_defines.append("\n")

    scalar_defines.append(c_define_int_str("HEAD_MULT", HEAD_MULT))
    scalar_defines.append(c_define_int_str("HEAD_SHIFT", HEAD_SHIFT))
    scalar_defines.append(c_define_int_str("HEAD_ZERO_PT", HEAD_ZERO_PT))
    scalar_defines.append(c_fixed_define_int_str("HEAD_IN_SCALE", head_in_scale, SCALE_FRAC_BITS))
    scalar_defines.append("\n")

    scalar_defines.append(c_define_int_str("BN0_SHIFT", BN0_SHIFT))
    scalar_defines.append(c_define_int_str("BN1_SHIFT", BN1_SHIFT))

    scalar_defines.append(c_define_int_str("MULT_a", MULT_a))
    scalar_defines.append(c_define_int_str("SHIFT_a", SHIFT_a))
    scalar_defines.append(c_define_int_str("ZP_OUT_a", ZP_OUT_a))
    scalar_defines.append(c_define_int_str("MULT_soft", MULT_soft))
    scalar_defines.append("\n")

    scalar_defines.append(c_define_int_str("MULT_kv", MULT_kv))
    scalar_defines.append(c_define_int_str("MULT_v", MULT_v))
    scalar_defines.append(c_define_int_str("SHIFT_v", SHIFT_v))
    scalar_defines.append(c_define_int_str("ZP_OUT_v", ZP_OUT_v))
    scalar_defines.append("\n")

    scalar_defines.append(c_define_int_str("RATIO", RATIO))
    scalar_defines.append(c_define_int_str("OUT_RES_0", OUT_RES_0))
    scalar_defines.append(c_define_int_str("RATIO1", RATIO1))
    scalar_defines.append(c_define_int_str("OUT_RES_1", OUT_RES_1))
    scalar_defines.append(c_define_int_str("OUT_RES_1_SHIFT", OUT_RES_1_SHIFT))
    scalar_defines.append(c_define_int_str("OUT_RES_0_SHIFT", OUT_RES_0_SHIFT))
    scalar_defines.append(c_define_int_str("RATIO_SHIFT", RATIO_SHIFT))
    scalar_defines.append(c_define_int_str("RATIO1_SHIFT", RATIO1_SHIFT))
    scalar_defines.append("\n")

    scalar_defines.append("\n")

    scalar_defines_txt = "".join(scalar_defines)

    h_txt = f"""#ifndef VIT_PARAMS_H
#define VIT_PARAMS_H

#include <stddef.h>
#include <stdint.h>

/*
 * No floating-point parameters are exported.
 *
 * Former float scales are exported as int32_t Q{SCALE_FRAC_BITS}:
 *     real_value = value / 2^{SCALE_FRAC_BITS}
 *
 * BN parameters and attention scale are exported as int32_t Q{PARAM_FRAC_BITS}:
 *     real_value = value / 2^{PARAM_FRAC_BITS}
 */

{scalar_defines_txt}
extern const int32_t rm_b0[E];
extern const int32_t rs_b0[E];
extern const int32_t scale_b0[E];
extern const int32_t bias_b0[E];

extern const int32_t rm_b1[E];
extern const int32_t rs_b1[E];
extern const int32_t scale_b1[E];
extern const int32_t bias_b1[E];

extern const int32_t rm_bf[E];
extern const int32_t rs_bf[E];
extern const int32_t scale_bf[E];
extern const int32_t bias_bf[E];

extern const int8_t w_q_0[E * E];
extern const int8_t w_k_0[E * E];
extern const int8_t w_v_0[E * E];
extern const int32_t bias_q_0[E];
extern const int32_t bias_k_0[E];
extern const int32_t bias_v_0[E];
extern const int8_t head_0[E * E];
extern const int32_t bias_h_0[E];

extern const int8_t w_1_0[E * F];
extern const int8_t w_2_0[F * E];
extern const int32_t bias_1_0[F];
extern const int32_t bias_2_0[E];

extern const int8_t patch_kernels_0[E * IN_CHANNELS * PATCH * PATCH];
extern const int32_t patch_bias_0[E];
extern const int8_t cls_token_0[E];
extern const int8_t pos_enc_0[(T + 1) * E];

extern const int8_t clas_weights_0[E * OUT_CLS];
extern const int32_t clas_bias_0[OUT_CLS];

extern const int32_t bn0_mul[E];
extern const int32_t bn0_add[E];

extern const int32_t bn1_mul[E];
extern const int32_t bn1_add[E];

#endif
"""

    (out_dir_h / "vit_params.h").write_text(h_txt)

    # ---------------- write vit_params.c ----------------

    c_parts = ['#include "parameters/vit_params.h"\n\n']

    # No scalar constants are emitted here.
    # All scalar constants are #define macros in vit_params.h.

    # BN params exported as Q16 int32_t arrays.
    c_parts.append(c_fixed_array_str("rm_b0", bn1["rm"], PARAM_FRAC_BITS))
    c_parts.append(c_fixed_array_str("rs_b0", bn1["rs"], PARAM_FRAC_BITS))
    c_parts.append(c_fixed_array_str("scale_b0", bn1["scale"], PARAM_FRAC_BITS))
    c_parts.append(c_fixed_array_str("bias_b0", bn1["bias"], PARAM_FRAC_BITS))
    c_parts.append("\n")

    c_parts.append(c_fixed_array_str("rm_b1", bn2["rm"], PARAM_FRAC_BITS))
    c_parts.append(c_fixed_array_str("rs_b1", bn2["rs"], PARAM_FRAC_BITS))
    c_parts.append(c_fixed_array_str("scale_b1", bn2["scale"], PARAM_FRAC_BITS))
    c_parts.append(c_fixed_array_str("bias_b1", bn2["bias"], PARAM_FRAC_BITS))
    c_parts.append("\n")

    c_parts.append(c_fixed_array_str("rm_bf", bn_final["rm"], PARAM_FRAC_BITS))
    c_parts.append(c_fixed_array_str("rs_bf", bn_final["rs"], PARAM_FRAC_BITS))
    c_parts.append(c_fixed_array_str("scale_bf", bn_final["scale"], PARAM_FRAC_BITS))
    c_parts.append(c_fixed_array_str("bias_bf", bn_final["bias"], PARAM_FRAC_BITS))
    c_parts.append("\n")

    c_parts.append(c_array_str("bn0_mul", bn0_mul, "int32_t"))
    c_parts.append(c_array_str("bn0_add", bn0_add, "int32_t"))
    c_parts.append("\n")

    c_parts.append(c_array_str("bn1_mul", bn1_mul, "int32_t"))
    c_parts.append(c_array_str("bn1_add", bn1_add, "int32_t"))
    c_parts.append("\n")
    

    arrays = [
        ("w_q_0", w_q_0, "int8_t"),
        ("w_k_0", w_k_0, "int8_t"),
        ("w_v_0", w_v_0, "int8_t"),
        ("bias_q_0", bias_q_0, "int32_t"),
        ("bias_k_0", bias_k_0, "int32_t"),
        ("bias_v_0", bias_v_0, "int32_t"),
        ("head_0", head_0, "int8_t"),
        ("bias_h_0", bias_h_0, "int32_t"),
        ("w_1_0", w_1_0, "int8_t"),
        ("w_2_0", w_2_0, "int8_t"),
        ("bias_1_0", bias_1_0, "int32_t"),
        ("bias_2_0", bias_2_0, "int32_t"),
        ("patch_kernels_0", patch_w, "int8_t"),
        ("patch_bias_0", patch_b, "int32_t"),
        ("cls_token_0", cls_token, "int8_t"),
        ("pos_enc_0", pos_enc, "int8_t"),
        ("clas_weights_0", clas_weights_0, "int8_t"),
        ("clas_bias_0", clas_bias_0, "int32_t"),

    ]

    for name, arr, ctype in arrays:
        c_parts.append(c_array_str(name, arr, ctype))
        c_parts.append("\n")

    (out_dir_c / "vit_params.c").write_text("".join(c_parts))

    print(f"Exported to: {out_dir_h / 'vit_params.h'}")
    print(f"Exported to: {out_dir_c / 'vit_params.cpp'}")

    return bn0_mul, bn0_add

    


model.eval()

bn0_mul, bn0_add = export_vit_to_c(
    model,
    config,
    out_dir_h="../../hls/include/parameters",
    out_dir_c="../../hls/src/parameters",
    verbose=True,
    test_loader=test_loader,
    device="cpu",
)


/home/fede/miniconda3/envs/ptorch/lib/python3.9/site-packages/torch/_tensor.py:1413: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at /opt/conda/conda-bld/pytorch_1720538643151/work/c10/core/TensorImpl.h:1925.)
  return super().rename(names)


S_out_proh_in: 0.019370974972844124
=== Export scales ===
INPUT_SCALE 0.004888935945928097
patch_out_scale 0.0035754956770688295
patch_weight_scale 0.0021143918856978416
bn1_out_scale 0.047797176986932755
bn2_out_scale 0.04356326535344124
q_out_scale 0.04329326003789902
k_out_scale 0.035507552325725555
v_out_scale 0.03309505432844162
soft_out_scale 0.0034119882620871067
out_proj_in_scale 0.019370974972844124
out_proj_out_scale 0.009948955848813057
fc1_out_scale 0.03323349729180336
fc2_in_scale 0.03323349729180336
fc2_out_scale 0.014436868950724602
head_in_scale 0.04356326535344124
BN0 MULT: [23582 15552 22971 19066 29378 19224 19766 23270 28599 23868 24043 17834
 26616 19677 20373 26551 22935 23983 26643 18674 24692 18713 24984 18857
 19255 18901 21820 20786 26542 20584 29801 21082 28493 24614 24476 19166
 24961 18319 16680 16553 25152 15242 17861 25055 21251 24627 20822 27087
 22141 24974 22875 21881 26214 20368 20988 22415 18952 22776 24743 22315
 23798 27102 26141 20568 28282 18109 

In [18]:
import torch

activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

targets = ["blocks.0", "blocks.0.mha.softmax","patch", 
           "blocks.0.mha","blocks.0.mha.v_proj",
           "blocks.0.mha.q_proj","blocks.0.mha.k_proj",
           "blocks.0.mha.out_proj", "blocks.0.mlp", 
           "norm", "blocks.0.bn1", "blocks.0.bn2",
           "blocks.0.mha.logit_quant","blocks.0.mha.soft_quant",
           "blocks.0.mlp.fc1", "blocks.0.quant_res0", "blocks.0.quant_res1",
           "blocks.0.quant_bn2_out", "blocks.0.mlp.fc2"]

for name, module in model.named_modules():
    if name in targets:
        module.register_forward_hook(get_activation(name))

model.to("cpu")
#def test(model, dataloader, criterion, device):
model.eval()

with torch.no_grad():
    images, labels = next(iter(test_loader))   # images: [B, 1, 28, 28]
    # Optionally pick just the first sample:
    img = images[0].unsqueeze(0)              # [1, 1, 28, 28]
    lbl = labels[0].unsqueeze(0)              # [1]

    outputs = model(img)
    loss = criterion(outputs, lbl)

    print(f"Image, Label: {lbl}, Output: {outputs}, Loss: {loss.item()}")

Image_size = config["image_size"]
Patch_size = config["patch_size"]

N = int((Image_size / Patch_size))**2 + 1
H = config["num_heads"]
H_dim = int(config["embed_dim"] / H)
print("img shape:", img.shape)

outputs_cpu = outputs.cpu().numpy()

img_cpu = img.permute(0, 2, 3, 1).cpu().numpy().flatten()
lbl_cpu = lbl.cpu().numpy()

patch_o = activations["patch"].cpu()
patch_o_int = patch_o

patch_o_int = patch_o.value / patch_o.scale

patch_o_int = patch_o_int.round()


patch_o_int = patch_o_int.flatten()

val = patch_o.value
blocco = model.blocks[0]
out_bn = blocco.quant_bn1_out(blocco._bn(blocco.bn1, val))


patch_o = val.numpy().flatten()



patch_o_tp = activations["patch"].cpu()
val = patch_o_tp.value
patch_o_tp = val.numpy().transpose(0, 2, 1).flatten()
norm_patch_o = activations["blocks.0.bn1"].cpu().numpy().transpose(0, 2, 1).flatten()  
norm_patch_2_o = activations["blocks.0.bn2"].cpu().numpy().transpose(0, 2, 1).flatten()
mha_o = activations["blocks.0.mha"].cpu().flatten()
mlp_o = activations["blocks.0.mlp"].cpu().flatten()
q_linear = activations["blocks.0.mha.q_proj"].cpu().view(1, N, H, H_dim).transpose(1, 2)#.contiguous().flatten()
v_linear = activations["blocks.0.mha.v_proj"].cpu().view(1, N, H, H_dim).transpose(1, 2)#.contiguous().flatten()
k_linear = activations["blocks.0.mha.k_proj"].cpu().view(1, N, H, H_dim).transpose(1, 2)#.contiguous()#.flatten()
out_linear = activations["blocks.0.mha.out_proj"].cpu()#.contiguous()#.flatten()

logit_quant = activations["blocks.0.mha.logit_quant"].cpu()
print("soft_quant = ", activations["blocks.0.mha.soft_quant"].cpu().scale * 127 * (1<<16))
q_scale = q_linear.scale
k_scale = k_linear.scale
l_scale = logit_quant.scale
print("v scale = ", v_linear.scale)
print("Mult_kv_scale = ", (1<<16) * out_linear.scale / (activations["blocks.0.mha.soft_quant"].cpu().scale * 127 *v_linear.scale))
S_scale = q_scale * k_scale * (48**(-0.5))
print("S_SCALE=", S_scale)
print("Q_SCALE=", q_scale)
print("K_SCALE=", k_scale)
print("L_SCALE=", l_scale)
print("RAD_SCALE=", 48**-0.5)
print("/////////////////")
print("MULT_a=", int(S_scale / l_scale * (1<<16)))
S_soft = float(activations["blocks.0.mha.soft_quant"].scale.reshape(-1)[0])
S_v = float(activations["blocks.0.mha.v_proj"].scale.reshape(-1)[0])

# Important: this must be out_proj INPUT quant scale, not out_proj output scale.
S_out_proj_in = 0.019370974972844124

MULT_kv = int(round((S_soft * S_v / S_out_proj_in) * (1 << 16)))

print("S_soft:", S_soft)
print("S_v:", S_v)
print("S_out_proj_in:", S_out_proj_in)
print("MULT_kv:", MULT_kv)

S_soft = float(activations["blocks.0.mha.soft_quant"].scale.reshape(-1)[0])

MULT_soft = int(round((1.0 / (127.0 * S_soft)) * (1 << 16)))

print("MULT_soft:", MULT_soft)

#softmax_module = activations["blocks.0.mha.softmax"].cpu().numpy().flatten()
q_linear_np = q_linear.value.detach().cpu()
k_linear_tp = k_linear.transpose(-2,-1)

k_linear_np = k_linear.value.detach().cpu()
k_linear_np_tp = k_linear_tp.value.detach().cpu()
v_linear_np = v_linear.value.detach().cpu()

q_linear = q_linear.value.numpy()
k_linear_tp = k_linear_tp.value.numpy()

matrix_a = q_linear @ k_linear_tp
matrix_a = matrix_a / (48**0.5)
matrix_a = torch.from_numpy(matrix_a)
matrix_a = F.softmax(matrix_a, dim=-1)

matrix_mha = matrix_a @ v_linear


np.savetxt("mnist_output.txt", outputs_cpu.reshape(-1, 1), fmt="%.16f")
np.savetxt("mnist_input.txt", img_cpu, fmt="%.16f")
np.savetxt("mnist_label.txt", lbl_cpu, fmt="%.16f")
np.savetxt("patch_output.txt", patch_o, fmt="%.16f")
np.savetxt("patch_output_int.txt", patch_o_int, fmt="%d")
np.savetxt("patch_output_tp.txt", patch_o_tp, fmt="%.16f")
np.savetxt("norm_0.txt", norm_patch_o, fmt="%.16f")
np.savetxt("norm_1.txt", norm_patch_2_o, fmt="%.16f")
np.savetxt("mha_0.txt", mha_o.value, fmt="%.16f")
np.savetxt("q.txt", q_linear_np.flatten(), fmt="%.16f")
np.savetxt("q_int.txt", (q_linear_np.flatten())/ q_scale, fmt="%d")

np.savetxt("v.txt", v_linear_np.flatten(), fmt="%.16f")
#np.savetxt("softmax_logits.txt", softmax_module, fmt="%.16f")
np.savetxt("k.txt", k_linear_np.flatten(), fmt="%.16f")
np.savetxt("k_int.txt", (k_linear_np.flatten())/ k_scale, fmt="%d")
np.savetxt("k_tp.txt", k_linear_np_tp.flatten(), fmt="%.16f")
np.savetxt("a.txt", matrix_a.flatten(), fmt="%.16f")



Image, Label: tensor([7]), Output: tensor([[-2.4011, -3.2265,  1.5007,  0.5253, -6.3030,  3.6768, -6.4531,  9.5296,
         -1.5007, -1.0505]]), Loss: 0.003366281744092703
img shape: torch.Size([1, 1, 32, 32])
soft_quant =  tensor(28398.2246)
v scale =  tensor(0.0331)
Mult_kv_scale =  tensor(45465.6172)
S_SCALE= tensor(0.0002)
Q_SCALE= tensor(0.0433)
K_SCALE= tensor(0.0355)
L_SCALE= tensor(0.1059)
RAD_SCALE= 0.14433756729740643
/////////////////
MULT_a= 137
S_soft: 0.0034119882620871067
S_v: 0.03309505432844162
S_out_proj_in: 0.019370974972844124
MULT_kv: 382
MULT_soft: 151241


In [19]:
import numpy as np
import torch

def value(x):
    return x.value if hasattr(x, "value") else x

def quant_int(x, signed=True, bit_width=8):
    if hasattr(x, "value") and hasattr(x, "scale"):
        v = x.value
        s = x.scale
        zp = x.zero_point if hasattr(x, "zero_point") and x.zero_point is not None else 0.0
        q = torch.round(v / s + zp)
    else:
        q = torch.round(x)

    if signed:
        qmin = -(2 ** (bit_width - 1))
        qmax = 2 ** (bit_width - 1) - 1
    else:
        qmin = 0
        qmax = 2 ** bit_width - 1

    return torch.clamp(q, qmin, qmax).to(torch.int32)

def save_int_column(path, x):
    arr = x.detach().cpu().reshape(-1).numpy().astype(np.int32)
    np.savetxt(path, arr, fmt="%d")

def save_float_column(path, x):
    arr = x.detach().cpu().reshape(-1).numpy().astype(np.float32)
    np.savetxt(path, arr, fmt="%.16f")

def flatten_linear_layer_hls_order(x, Tm, Tn, PE):
    """
    x: Tensor shaped [M, N]
       Example Q/K/V projection output after quantization: [17, 192]

    Returns values in the same order as your HLS Linear_Layer debug printf:
        for m in range(0, M, Tm):
          for n in range(0, N, Tn):
            for tm in range(Tm):
              for tn in range(Tn // PE):
                for pe in range(PE):
                  print x[m + tm, n + tn * PE + pe]
    """
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()

    x = np.asarray(x)
    assert x.ndim == 2, f"Expected [M, N], got {x.shape}"

    M, N = x.shape

    assert M % Tm == 0, f"M={M} must be divisible by Tm={Tm}"
    assert N % Tn == 0, f"N={N} must be divisible by Tn={Tn}"
    assert Tn % PE == 0, f"Tn={Tn} must be divisible by PE={PE}"

    out = []

    for m in range(0, M, Tm):
        for n in range(0, N, Tn):
            for tm in range(Tm):
                for tn in range(Tn // PE):
                    for pe in range(PE):
                        out.append(x[m + tm, n + tn * PE + pe])

    return np.asarray(out)

N_tokens = N
E = config["embed_dim"]

# Put your actual HLS config values here
Tm = 1   # replace manually
Tn = 1   # replace manually
PE = 1   # replace manually

block_proj = activations["blocks.0"]      # QuantTensor [1, 17, 192]
block_proj_int = quant_int(block_proj)[0]                # [17, 192]

block_ref_hls_order = flatten_linear_layer_hls_order(
    block_proj_int,
    Tm=Tm,
    Tn=Tn,
    PE=PE,
)

v_proj = activations["blocks.0.mha.v_proj"]      # QuantTensor [1, 17, 192]
v_proj_int = quant_int(v_proj)[0]                # [17, 192]

v_ref_hls_order = flatten_linear_layer_hls_order(
    v_proj_int,
    Tm=Tm,
    Tn=Tn,
    PE=PE,
)

k_proj = activations["blocks.0.mha.k_proj"]      # QuantTensor [1, 17, 192]
k_proj_int = quant_int(k_proj)[0]                # [17, 192]

k_ref_hls_order = flatten_linear_layer_hls_order(
    k_proj_int,
    Tm=Tm,
    Tn=Tn,
    PE=PE,
)

q_proj = activations["blocks.0.mha.q_proj"]      # QuantTensor [1, 17, 192]
q_proj_int = quant_int(q_proj)[0]                # [17, 192]

q_ref_hls_order = flatten_linear_layer_hls_order(
    q_proj_int,
    Tm=Tm,
    Tn=Tn,
    PE=PE,
)

out_proj = activations["blocks.0.mha.out_proj"]      # QuantTensor [1, 17, 192]
out_proj_int = quant_int(out_proj)[0]                # [17, 192]

out_ref_hls_order = flatten_linear_layer_hls_order(
    out_proj_int,
    Tm=Tm,
    Tn=Tn,
    PE=PE,
)

fc1_proj = activations["blocks.0.mlp.fc1"]      # QuantTensor [1, 17, 192]
fc1_proj_int = quant_int(fc1_proj)[0]                # [17, 192]

fc1_proj_hls_order = flatten_linear_layer_hls_order(
    fc1_proj_int,
    Tm=Tm,
    Tn=Tn,
    PE=PE,
)

fc2_proj = activations["blocks.0.mlp.fc2"]      # QuantTensor [1, 17, 192]
fc2_proj_int = quant_int(fc2_proj)[0]                # [17, 192]

fc2_proj_hls_order = flatten_linear_layer_hls_order(
    fc2_proj_int,
    Tm=Tm,
    Tn=Tn,
    PE=PE,
)


patch_o = activations["patch"].cpu()
requant_patch_0 = patch_o * (patch_o.scale / out_proj.scale)

S_out = (float)(out_proj.scale)
S_patch = (float)(patch_o.scale)


print("S_out:", S_out)
print("S_patch:", S_patch)
print("S_x = S_patch / S_out =", S_patch / S_out)
print("MULT_x = S_x * 2**16 =", round((S_patch / S_out)*(1<<16)))

res_0 = torch.round(patch_o / patch_o.scale)

res_0 = torch.round(res_0 * (patch_o.scale / out_proj.scale))

res_0 = res_0 + (torch.round(out_proj / out_proj.scale))

res_0 = torch.clamp(res_0, -128,127)

np.savetxt("ref_out_proj_hls_order_int.txt", out_ref_hls_order, fmt="%d")
np.savetxt("ref_v_proj_hls_order_int.txt", v_ref_hls_order, fmt="%d")
np.savetxt("ref_block_proj_hls_order_int.txt", block_ref_hls_order, fmt="%d")
np.savetxt("ref_q_proj_hls_order_int.txt", q_ref_hls_order, fmt="%d")
np.savetxt("ref_k_proj_hls_order_int.txt", k_ref_hls_order, fmt="%d")
np.savetxt("ref_fc1_proj_hls_order_int.txt", fc1_proj_hls_order, fmt="%d")

np.savetxt("ref_res_0_hls_order_int.txt", res_0.flatten(), fmt="%d")

quant_res1 = activations["blocks.0.quant_res1"].cpu()
quant_res0 = activations["blocks.0.quant_res0"].cpu()

print("RES_1", int((patch_o.scale/ quant_res1.scale) * (1<<16)))
print("RES_0", int((out_proj.scale/ quant_res0.scale)* (1<<16)))

res_0 = float(activations["blocks.0.quant_res0"].scale) 
res_1 = float(activations["blocks.0.quant_res1"].scale) 
fc2_out_proj_scale = float(activations["blocks.0.mlp.fc2"].scale)
print("res 0:", float(activations["blocks.0.quant_res0"].scale))
print("second requant:",int( res_0 / fc2_out_proj_scale * (1<<16)))

print("last quant residual:",int( fc2_out_proj_scale / res_1 * (1<<16))), 

#print("RES_second_branch:", int(()))

print("res_before_add:", int((patch_o.scale / out_proj.scale)*(1<<16)))

RATIO1 = round((res_0 / fc2_out_proj_scale) * (1 << 16))
OUT_RES_1 = round((fc2_out_proj_scale / res_1) * (1 << 16))

print(f"RATIO1: {RATIO1} ---- OUT_RES_1: {OUT_RES_1}")


S_out: 0.009948955848813057
S_patch: 0.0035754956770688295
S_x = S_patch / S_out = 0.3593840128957249
MULT_x = S_x * 2**16 = 23553
RES_1 8989
RES_0 43293
res 0: 0.015060232020914555
second requant: 68365
last quant residual: 36296
res_before_add: 23552
RATIO1: 68366 ---- OUT_RES_1: 36296


In [21]:
def compare_hls_to_ref(hls_path, ref, dtype=np.int32, atol=0, rtol=0, topk=20):
    hls = np.loadtxt(hls_path, dtype=dtype).reshape(-1)

    if isinstance(ref, torch.Tensor):
        ref = ref.detach().cpu().reshape(-1).numpy()
    else:
        ref = np.asarray(ref).reshape(-1)

    if hls.shape != ref.shape:
        raise ValueError(f"Shape mismatch: HLS {hls.shape}, REF {ref.shape}")

    diff = hls.astype(np.float64) - ref.astype(np.float64)

    print("num values:", ref.size)
    print("max abs diff:", np.max(np.abs(diff)))
    print("mean abs diff:", np.mean(np.abs(diff)))
    print("allclose:", np.allclose(hls, ref, atol=atol, rtol=rtol))

    worst = np.argsort(np.abs(diff))[-topk:][::-1]

    print("\nWorst mismatches:")
    for i in worst:
        print(
            i,
            "hls =", hls[i],
            "ref =", ref[i],
            "diff =", diff[i],
        )

compare_hls_to_ref(
    "hls_block_output.txt",
    block_ref_hls_order,
    dtype=np.int32,
    atol=0,
)



num values: 3264
max abs diff: 275.0
mean abs diff: 139.7996323529412
allclose: False

Worst mismatches:
353 hls = 252 ref = -23 diff = 275.0
347 hls = 238 ref = -35 diff = 273.0
2856 hls = 243 ref = -28 diff = 271.0
923 hls = 228 ref = -40 diff = 268.0
2329 hls = 240 ref = -28 diff = 268.0
1185 hls = 213 ref = -53 diff = 266.0
2993 hls = 211 ref = -55 diff = 266.0
273 hls = 223 ref = -43 diff = 266.0
335 hls = 192 ref = -74 diff = 266.0
929 hls = 238 ref = -27 diff = 265.0
523 hls = 223 ref = -42 diff = 265.0
863 hls = 250 ref = -15 diff = 265.0
2732 hls = 253 ref = -11 diff = 264.0
911 hls = 193 ref = -71 diff = 264.0
1284 hls = 191 ref = -73 diff = 264.0
409 hls = 224 ref = -39 diff = 263.0
2405 hls = 216 ref = -47 diff = 263.0
1966 hls = 190 ref = -73 diff = 263.0
2746 hls = 248 ref = -15 diff = 263.0
454 hls = 235 ref = -28 diff = 263.0


/tmp/ipykernel_835338/883368892.py:2: DeprecationWarning: loadtxt(): Parsing an integer via a float is deprecated.  To avoid this warning, you can:
    * make sure the original data is stored as integers.
    * use the `converters=` keyword argument.  If you only use
      NumPy 1.23 or later, `converters=float` will normally work.
    * Use `np.loadtxt(...).astype(np.int64)` parsing the file as
      floating point and then convert it.  (On all NumPy versions.)
  (Deprecated NumPy 1.23)
  hls = np.loadtxt(hls_path, dtype=dtype).reshape(-1)


In [29]:
res0_ref = quant_int(activations["blocks.0.quant_res0"])[0]
bn2_ref  = quant_int(activations["blocks.0.quant_bn2_out"])[0]
fc1_ref  = quant_int(activations["blocks.0.mlp.fc1"])[0]

bn2_ref_hls_order = flatten_linear_layer_hls_order(
    bn2_ref,
    Tm=1,
    Tn=1,
    PE=1,
)

compare_hls_to_ref(
    "batch_1_output.txt",
    bn2_ref_hls_order,
    dtype=np.int32,
    atol=0,
)

num values: 3264
max abs diff: 5.0
mean abs diff: 0.5756740196078431
allclose: False

Worst mismatches:
2854 hls = -3 ref = -8 diff = 5.0
2875 hls = 32 ref = 28 diff = 4.0
3064 hls = 1 ref = 5 diff = -4.0
135 hls = 22 ref = 18 diff = 4.0
2823 hls = 55 ref = 51 diff = 4.0
183 hls = 15 ref = 19 diff = -4.0
264 hls = -30 ref = -33 diff = 3.0
295 hls = 27 ref = 30 diff = -3.0
2385 hls = -19 ref = -16 diff = -3.0
2879 hls = -7 ref = -4 diff = -3.0
3023 hls = 0 ref = -3 diff = 3.0
2877 hls = -78 ref = -81 diff = 3.0
2320 hls = 14 ref = 11 diff = 3.0
3139 hls = -34 ref = -31 diff = -3.0
2325 hls = -34 ref = -31 diff = -3.0
2448 hls = 17 ref = 14 diff = 3.0
1962 hls = 28 ref = 25 diff = 3.0
2732 hls = -2 ref = -5 diff = 3.0
2376 hls = -18 ref = -15 diff = -3.0
1967 hls = -30 ref = -33 diff = 3.0


/tmp/ipykernel_716007/260335103.py:2: DeprecationWarning: loadtxt(): Parsing an integer via a float is deprecated.  To avoid this warning, you can:
    * make sure the original data is stored as integers.
    * use the `converters=` keyword argument.  If you only use
      NumPy 1.23 or later, `converters=float` will normally work.
    * Use `np.loadtxt(...).astype(np.int64)` parsing the file as
      floating point and then convert it.  (On all NumPy versions.)
  (Deprecated NumPy 1.23)
  hls = np.loadtxt(hls_path, dtype=dtype).reshape(-1)


In [25]:
model.eval()
model.cpu()

array = np.loadtxt("./hls_block_output.txt", dtype=np.float32)
out_block_int = torch.from_numpy(array).reshape(1, 17, 192)

out_block_ref = activations["blocks.0"]

S_block = float(out_block_ref.scale.reshape(-1)[0])

out_block_real = out_block_int * S_block

with torch.no_grad():
    x = model._bn(model.norm, out_block_real)
    cls_token = x[:, 0, :]
    output_hls = model.head(cls_token)
    if hasattr(output_hls, "value"):
        output_hls = output_hls.value

    x = model._bn(model.norm, out_block_ref.value)
    cls_token = x[:, 0, :]
    output_ref = model.head(cls_token)
    if hasattr(output_ref, "value"):
        output_ref = output_ref.value

print(output_hls)
print(output_ref)

tensor([[-2.4011, -3.3766,  1.5007,  0.5253, -6.3030,  3.7518, -6.5281,  9.5296,
         -1.4257, -0.9755]])
tensor([[-2.4011, -3.2265,  1.5007,  0.5253, -6.3030,  3.6768, -6.4531,  9.5296,
         -1.5007, -1.0505]])


In [ ]:
from export_vit_hls_params import export_vit_to_hls

hls_cfg = {
    "debug": False,
    "types": {
        "axi_width": 8,
        "act_width": 8,
        "act_signed": True,
        "weight_width": 8,
        "weight_signed": True,
        "acc_width": 32,
        "acc_signed": True,
    },
    "linear_cfg": {
        "base": {"SIMD": 1, "PE": 1, "Tm": 1, "Tk": 1, "Tn": 1},
        "fc1":  {"SIMD": 1, "PE": 1, "Tm": 1, "Tk": 1, "Tn": 1},
        "fc2":  {"SIMD": 1, "PE": 1, "Tm": 1, "Tk": 1, "Tn": 1},
    },
}

export_vit_to_hls(
    model,
    config,
    out_dir_h="../../hls/include/parameters",
    out_dir_cpp="../../hls/src/parameters",
    hls_export_config=hls_cfg,
    verbose=True,
    test_loader=test_loader,
    device="cpu",
)

=== Export scales ===
INPUT_SCALE 0.004888935945928097
patch_out_scale 0.0035754956770688295
patch_weight_scale 0.0021143918856978416
bn1_out_scale 0.047797176986932755
bn2_out_scale 0.04356326535344124
q_out_scale 0.04329326003789902
k_out_scale 0.035507552325725555
v_out_scale 0.03309505432844162
soft_out_scale 0.0034119882620871067
out_proj_in_scale 0.019370974972844124
out_proj_out_scale 0.009948955848813057
fc1_out_scale 0.03323349729180336
fc2_in_scale 0.03323349729180336
fc2_out_scale 0.014436868950724602
head_in_scale 0.04356326535344124

=== Exported tensor sanity ===
patch_kernels_0
  shape: (192, 1, 8, 8)
  min/max: -127 116
  nonzero count: 12183
  first 20: [ 22 -29 -65  25 -42  -3 -28  41  -3 -34 -58  52   4  36 -67 -30 -37 -13
  34  -4]
patch_bias_0
  shape: (192,)
  min/max: -29423 21458
  nonzero count: 192
  first 20: [-13975  20658   9100  -2079   -349  -7573   2021  13242  -6694  -9561
  -7593 -19652  17182 -11854  -2207 -11053  18759   7071   7572  17237]
w_q_0
  s

{'types_h': PosixPath('../../hls/include/types.h'),
 'vit_params_h': PosixPath('../../hls/include/vit_params.h'),
 'vit_params_cpp': PosixPath('../../hls/src/parameters/vit_params.cpp'),
 'bn0_mul': array([23582, 15552, 22971, 19066, 29378, 19224, 19766, 23270, 28599,
        23868, 24043, 17834, 26616, 19677, 20373, 26551, 22935, 23983,
        26643, 18674, 24692, 18713, 24984, 18857, 19255, 18901, 21820,
        20786, 26542, 20584, 29801, 21082, 28493, 24614, 24476, 19166,
        24961, 18319, 16680, 16553, 25152, 15242, 17861, 25055, 21251,
        24627, 20822, 27087, 22141, 24974, 22875, 21881, 26214, 20368,
        20988, 22415, 18952, 22776, 24743, 22315, 23798, 27102, 26141,
        20568, 28282, 18109, 19235, 25417, 21248, 19794, 21667, 19338,
        23457, 22150, 21493, 24459, 19400, 27427, 23603, 23547, 23043,
        19617, 18613, 21123, 23326, 19001, 19421, 25252, 21371, 19250,
        24547, 28286, 15761, 23491, 22319, 19133, 20554, 17189, 23062,
        21170, 27200,

In [ ]:
from brevitas.export import export_qonnx
from brevitas.quant import Int8ActPerTensorFloat, Int16Bias

float_inp = torch.randn(1, 1,28,28)

model.to("cpu")
model.eval()

output_path = './qvit_onnx_int8.onnx'
exported_model = export_qonnx(model, input_t=float_inp, export_path=output_path)